# Crypto Assembly Forecaster — Colab Training Notebook

Trains `CryptoAssemblyForecaster` (GRU + LightGBM + N-HiTS → Ridge) on Colab GPU.

**Before running:** Runtime → Change runtime type → T4 GPU.

## 1. Check GPU

In [31]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), 'GB')
else:
    print('No GPU — go to Runtime > Change runtime type > GPU')

CUDA available: True
GPU: Tesla T4
VRAM: 15.64 GB


## 2. Install dependencies

In [39]:
%%capture
!pip install neuralforecast>=1.7.0 lightgbm>=4.0.0 yfinance joblib scikit-learn scipy chronos-forecasting

## 3. Mount Google Drive and clone/pull the repo

In [33]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [34]:
import os, sys

REPO_DIR    = '/content/drive/MyDrive/capstone_project_unfc'
REPO_URL    = 'https://github.com/RocioT08/capstone_project_unfc.git'
BACKEND_DIR = os.path.join(REPO_DIR, 'backend')

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

if BACKEND_DIR not in sys.path:
    sys.path.insert(0, BACKEND_DIR)
print('Backend dir:', BACKEND_DIR)

You are not currently on a branch.
Please specify which branch you want to merge with.
See git-pull(1) for details.

    git pull <remote> <branch>

Backend dir: /content/drive/MyDrive/capstone_project_unfc/backend


## 4. Configuration

In [35]:
import random
import numpy as np
import torch

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

# ── Tickers ───────────────────────────────────────────────────────────────────
CRYPTO_TICKERS = [
    'BTC-USD',
    # 'ETH-USD',
    # 'BNB-USD',
    # 'SOL-USD',
    # 'XRP-USD',
    # 'ADA-USD',
    # 'AVAX-USD',
    # 'DOGE-USD',
]

TICKERS_WITH_FEAR_GREED = {
    'ETH-USD', 'BNB-USD', 'SOL-USD', 'XRP-USD',
    'ADA-USD', 'AVAX-USD', 'DOGE-USD',
}

DATA_START       = '2016-01-01'
CONFIDENCE_LEVEL = 0.95
CHECKPOINTS_DIR  = '/content/drive/MyDrive/capstone_checkpoints'
os.makedirs(CHECKPOINTS_DIR, exist_ok=True)

HOLDOUT_REGIMES = [
    '2017-12-17',  # ATH bull peak (~$20k)
    '2018-12-15',  # bear market bottom (~$3k)
    '2020-03-12',  # COVID crash (Black Thursday)
    '2021-11-10',  # ATH 2021 (~$69k)
    '2022-11-11',  # FTX collapse
    '2024-04-20',  # BTC halving
    '2025-01-01',  # bull market start
    '2026-01-01',  # current regime
]

print('Data start :', DATA_START)
print('Checkpoints:', CHECKPOINTS_DIR)
print('Regimes    :', len(HOLDOUT_REGIMES))

Data start : 2016-01-01
Checkpoints: /content/drive/MyDrive/capstone_checkpoints
Regimes    : 8


## 5. Data helpers (yfinance)

In [36]:
import pandas as pd
import yfinance as yf
import math

def fetch_ohlcv_yf(symbol: str, start: str = DATA_START) -> pd.DataFrame:
    ticker = yf.Ticker(symbol)
    df = ticker.history(start=start, auto_adjust=True)
    df.index = pd.to_datetime(df.index, utc=True)
    df = df[['Open', 'High', 'Low', 'Close', 'Volume']].astype(float)
    df = df.sort_index().dropna()
    print(f'{symbol}: {len(df)} rows from {df.index[0].date()} to {df.index[-1].date()}')
    return df

def _compute_error_metrics(actuals, predictions):
    mae  = float(np.mean([abs(a - p) for a, p in zip(actuals, predictions)]))
    rmse = float(math.sqrt(np.mean([(a - p)**2 for a, p in zip(actuals, predictions)])))
    mape = float(np.mean([abs(a - p) / abs(a) * 100 for a, p in zip(actuals, predictions) if a != 0]))
    return {'mae': round(mae, 4), 'rmse': round(rmse, 4), 'mape': round(mape, 4)}

# Quick test
test_df = fetch_ohlcv_yf('BTC-USD')
print(test_df.tail(3))

BTC-USD: 3762 rows from 2016-01-01 to 2026-04-20
                                   Open          High           Low  \
Date                                                                  
2026-04-17 00:00:00+00:00  75164.039062  78320.679688  74558.601562   
2026-04-18 00:00:00+00:00  77136.046875  77416.703125  75504.945312   
2026-04-20 00:00:00+00:00  73820.109375  74736.390625  73820.109375   

                                  Close        Volume  
Date                                                   
2026-04-17 00:00:00+00:00  77126.875000  5.413719e+10  
2026-04-18 00:00:00+00:00  75726.210938  2.601442e+10  
2026-04-20 00:00:00+00:00  74305.000000  3.292546e+10  


## 6. Import models

In [37]:
from analytics.forecasting.crypto.assembly import CryptoAssemblyForecaster
from analytics.forecasting.crypto.nhits_forecaster import _fetch_fear_greed
print('Imports OK')

Imports OK


## 7. Train all tickers (opcional — solo para guardar modelos en Drive)

In [ ]:
import joblib
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')

results_summary = {}

for symbol in CRYPTO_TICKERS:
    print(f"\n{'='*60}\nTraining: {symbol}\n{'='*60}")
    try:
        ohlcv = fetch_ohlcv_yf(symbol)
    except Exception as e:
        print(f'  [ERROR] fetch: {e}'); continue

    fear_greed = None
    if symbol in TICKERS_WITH_FEAR_GREED:
        try:
            fear_greed = _fetch_fear_greed(n_days=3000)
            print(f'  Fear & Greed: {len(fear_greed)} rows')
        except Exception as e:
            print(f'  Fear & Greed unavailable: {e}')

    model = CryptoAssemblyForecaster(
        max_horizon=7, n_splits=4, ridge_alpha=1.0, min_train_size=120,
        confidence_level=CONFIDENCE_LEVEL, use_gru=True, use_tft=False,
        gru_kwargs={'epochs': 30, 'mc_samples': 50, 'lookback': 60},
        nhits_kwargs={'max_steps': 1000},
        lgb_kwargs={'n_estimators': 300},
    )
    try:
        model.fit(ohlcv, fear_greed=fear_greed)
    except Exception as e:
        print(f'  [ERROR] fit: {e}'); continue

    ckpt = os.path.join(CHECKPOINTS_DIR, f'assembly_{symbol}.joblib')
    joblib.dump(model, ckpt)
    print(f'  Saved → {ckpt}')
    result = model.forecast(periods=7)
    print(f'  7-day forecast: {[round(p, 2) for p in result["point_forecast"]]}')
    results_summary[symbol] = 'ok'

print('\nDone:', list(results_summary.keys()))

## 8. Assembly — regime evaluation

In [38]:
eval_symbol = 'BTC-USD'
ohlcv = fetch_ohlcv_yf(eval_symbol)

fear_greed = None
if eval_symbol in TICKERS_WITH_FEAR_GREED:
    try:
        fear_greed = _fetch_fear_greed(n_days=3000)
    except Exception:
        pass

regime_results = []

for regime_date in HOLDOUT_REGIMES:
    cutoff = pd.Timestamp(regime_date)
    if ohlcv.index.tz is not None:
        cutoff = cutoff.tz_localize(ohlcv.index.tz)

    train_ohlcv = ohlcv[ohlcv.index <= cutoff]
    test_ohlcv  = ohlcv[ohlcv.index >  cutoff]

    if len(train_ohlcv) < 200 or len(test_ohlcv) < 7:
        print(f'Skipping {regime_date}: insufficient data'); continue

    test_prices = test_ohlcv['Close'].values[:7]
    print(f'\nAssembly regime {regime_date}: train={len(train_ohlcv)} rows')

    m = CryptoAssemblyForecaster(
        max_horizon=7, n_splits=4, ridge_alpha=1.0, min_train_size=120,
        confidence_level=CONFIDENCE_LEVEL, use_gru=True, use_tft=False,
        gru_kwargs={'epochs': 50, 'mc_samples': 50, 'lookback': 60,
                    'hidden_size': 128, 'num_layers': 3
                    },
        nhits_kwargs={'max_steps': 500, 'input_size': 120},
        lgb_kwargs={'lags': 28, 'n_estimators': 500,
                    'learning_rate': 0.03, 'num_leaves': 63
                    },
    )
    m.fit(train_ohlcv, fear_greed=fear_greed)
    result = m.forecast(periods=7)

    metrics = _compute_error_metrics(test_prices.tolist(), result['point_forecast'])
    print(f'  MAE={metrics["mae"]:.4f}  RMSE={metrics["rmse"]:.4f}  MAPE={metrics["mape"]:.4f}%')
    regime_results.append({'regime': regime_date, **metrics})

print('\n=== Assembly Regime Summary ===')
df_results = pd.DataFrame(regime_results)
print(df_results.to_string(index=False))
print('\nAssembly Mean MAPE:', round(df_results['mape'].mean(), 4), '%')

BTC-USD: 3762 rows from 2016-01-01 to 2026-04-20

Assembly regime 2017-12-17: train=717 rows


Install with: pip install neuralforecast
Traceback (most recent call last):
  File "/content/drive/MyDrive/capstone_project_unfc/backend/analytics/forecasting/crypto/assembly.py", line 469, in _fit_and_predict_fold
    nhits = NHiTSForecaster(
            ^^^^^^^^^^^^^^^^
  File "/content/drive/MyDrive/capstone_project_unfc/backend/analytics/forecasting/crypto/nhits_forecaster.py", line 255, in __init__
    raise ImportError(
ImportError: neuralforecast is required for NHiTSForecaster.
Install with: pip install neuralforecast
Install with: pip install neuralforecast
Traceback (most recent call last):
  File "/content/drive/MyDrive/capstone_project_unfc/backend/analytics/forecasting/crypto/assembly.py", line 469, in _fit_and_predict_fold
    nhits = NHiTSForecaster(
            ^^^^^^^^^^^^^^^^
  File "/content/drive/MyDrive/capstone_project_unfc/backend/analytics/forecasting/crypto/nhits_forecaster.py", line 255, in __init__
    raise ImportError(
ImportError: neuralforecast is require

ImportError: neuralforecast is required for NHiTSForecaster.
Install with: pip install neuralforecast

## 9. Chronos — regime evaluation (benchmark)-test

In [ ]:
from analytics.forecasting import chronos2

eval_symbol = 'BTC-USD'
ohlcv_c = fetch_ohlcv_yf(eval_symbol)

chronos_results = []

for regime_date in HOLDOUT_REGIMES:
    cutoff = pd.Timestamp(regime_date)
    if ohlcv_c.index.tz is not None:
        cutoff = cutoff.tz_localize(ohlcv_c.index.tz)

    train_prices = ohlcv_c['Close'][ohlcv_c.index <= cutoff]
    test_prices  = ohlcv_c['Close'][ohlcv_c.index >  cutoff].values[:7]

    if len(train_prices) < 200 or len(test_prices) < 7:
        print(f'Skipping {regime_date}: insufficient data'); continue

    print(f'\nChronos regime {regime_date}: {len(train_prices)} rows')
    try:
        result  = chronos2.forecast(train_prices, 7, CONFIDENCE_LEVEL, '1d')
        metrics = _compute_error_metrics(test_prices.tolist(), result['point_forecast'])
        print(f'  MAE={metrics["mae"]:.4f}  RMSE={metrics["rmse"]:.4f}  MAPE={metrics["mape"]:.4f}%')
        chronos_results.append({'regime': regime_date, **metrics})
    except Exception as e:
        print(f'  ERROR: {e}')

print('\n=== Chronos Regime Summary ===')
df_chronos = pd.DataFrame(chronos_results)
print(df_chronos.to_string(index=False))
print('\nChronos Mean MAPE:', round(df_chronos['mape'].mean(), 4), '%')

BTC-USD: 3761 rows from 2016-01-01 to 2026-04-19

Chronos regime 2017-12-17: 717 rows


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/478M [00:00<?, ?B/s]

  MAE=4380.7747  RMSE=4890.6786  MAPE=29.3406%

Chronos regime 2018-12-15: 1080 rows
  MAE=556.5321  RMSE=634.8817  MAPE=14.2872%

Chronos regime 2020-03-12: 1533 rows
  MAE=489.4399  RMSE=601.2610  MAPE=8.6966%

Chronos regime 2021-11-10: 2141 rows
  MAE=2146.7857  RMSE=3026.0536  MAPE=3.5035%

Chronos regime 2022-11-11: 2507 rows
  MAE=593.9978  RMSE=620.3941  MAPE=3.5727%

Chronos regime 2024-04-20: 3033 rows
  MAE=1019.6741  RMSE=1137.4347  MAPE=1.5692%

Chronos regime 2025-01-01: 3289 rows
  MAE=4409.7232  RMSE=4858.7375  MAPE=4.4624%

Chronos regime 2026-01-01: 3654 rows
  MAE=3231.2634  RMSE=3534.6067  MAPE=3.5008%

=== Chronos Regime Summary ===
    regime       mae      rmse    mape
2017-12-17 4380.7747 4890.6786 29.3406
2018-12-15  556.5321  634.8817 14.2872
2020-03-12  489.4399  601.2610  8.6966
2021-11-10 2146.7857 3026.0536  3.5035
2022-11-11  593.9978  620.3941  3.5727
2024-04-20 1019.6741 1137.4347  1.5692
2025-01-01 4409.7232 4858.7375  4.4624
2026-01-01 3231.2634 3534.

## 10. Assembly vs Chronos — comparison table

In [ ]:
df_compare = df_results.merge(df_chronos, on='regime', suffixes=('_assembly', '_chronos'))
df_compare['winner'] = df_compare.apply(
    lambda r: 'Assembly ✅' if r['mape_assembly'] < r['mape_chronos'] else 'Chronos ✅', axis=1
)
print('=== Assembly vs Chronos ===')
print(df_compare[['regime', 'mape_assembly', 'mape_chronos', 'winner']].to_string(index=False))
print(f'\nAssembly Mean MAPE : {df_compare["mape_assembly"].mean():.4f}%')
print(f'Chronos  Mean MAPE : {df_compare["mape_chronos"].mean():.4f}%')
wins = (df_compare['winner'] == 'Assembly ✅').sum()
print(f'Assembly wins      : {wins}/{len(df_compare)} regimes')

## 11. (Optional) Upload models to Supabase Storage

In [ ]:
# from google.colab import userdata
# SUPABASE_URL = userdata.get('SUPABASE_URL')
# SUPABASE_KEY = userdata.get('SUPABASE_KEY')
# from supabase import create_client
# db = create_client(SUPABASE_URL, SUPABASE_KEY)
# for symbol in CRYPTO_TICKERS:
#     ckpt = os.path.join(CHECKPOINTS_DIR, f'assembly_{symbol}.joblib')
#     if not os.path.exists(ckpt): continue
#     with open(ckpt, 'rb') as f:
#         db.storage.from_('models').upload(
#             f'assembly_{symbol}.joblib', f.read(),
#             file_options={'upsert': 'true', 'content-type': 'application/octet-stream'},
#         )
#     print(f'Uploaded {symbol}')